In [1]:
import json
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from virus_sim_dashboard import sim

In [2]:
%pwd

'/home/yinchi/github_projects/virus-sim-dashboard/notebooks'

In [3]:
data =json.load(open('../assets/private/main_store_data.json'))

env_factory = sim.EnvironmentFactory(data)
env_factory

## Step 1

In [4]:
data['step1']['disease_name']

'Influenza'

In [5]:
from base64 import b64decode
from io import BytesIO

pd.read_feather(BytesIO(b64decode(data['step1']['patient_stays'])))

,Age,Admission,Discharge,ICUAdmission,...,ReadmissionDischarge,FirstPosCollected,Acquisition,Summary
0,22,2016-08-03 16:50:02,2016-08-04 12:46:27.840,NaT,...,NaT,2016-08-03 22:54:09.408,Community-Onset Community-Associated,Discharged
1,84,2016-08-07 05:24:18,2016-08-10 17:19:21.600,NaT,...,NaT,2016-08-08 13:06:44.352,Community-Onset Community-Associated,Discharged
2,48,2016-08-31 00:22:02,2016-09-01 16:59:48.096,NaT,...,NaT,2016-08-31 15:43:44.128,Community-Onset Community-Associated,Discharged
3,22,2016-11-19 18:18:43,2016-11-21 18:39:04.704,NaT,...,NaT,2016-11-20 16:32:46.080,Community-Onset Community-Associated,Discharged
4,50,2016-11-24 14:32:56,2016-11-25 18:00:22.272,NaT,...,NaT,2016-11-25 17:07:56.544,Community-Onset Community-Associated,Discharged
...,...,...,...,...,...,...,...,...,...
4694,78,2025-05-07 21:44:00,2025-05-17 20:02:24.896,NaT,...,NaT,2025-05-15 23:59:07.776,Hospital-Onset Healthcare-Associated,Discharged
4695,91,2025-05-15 15:32:00,2025-05-27 10:43:37.088,NaT,...,NaT,2025-05-16 12:00:01.536,Community-Onset Community-Associated,Discharged
4696,84,2025-05-17 10:55:00,2025-05-19 15:06:43.072,NaT,...,NaT,2025-05-17 12:36:46.208,Community-Onset Community-Associated,Discharged
4697,7,2025-05-19 14:15:00,2025-05-19 18:12:24.192,NaT,...,2025-05-23 17:48:59.264,2025-05-19 14:14:17.344,Community-Onset Community-Associated,Discharged


## Step 2

In [6]:
print(json.dumps(data['step2'], indent=2))

{
  "start_opt_community": "Admission",
  "start_opt_other": "FirstPosCollected"
}


## Step 3

In [7]:
data['step3']['los_fit_results']['gim']['1']['los_fit'].keys()

dict_keys(['distribution', 'parameters', 'pp_plot_b64'])

In [8]:
display(data['step3']['los_fit_results']['icu']['1']['pre_icu_los_fit'].keys())
display(data['step3']['los_fit_results']['icu']['1']['icu_los_fit'].keys())
display(data['step3']['los_fit_results']['icu']['1']['post_icu_los_fit'].keys())

dict_keys(['distribution', 'parameters', 'pp_plot_b64'])

dict_keys(['distribution', 'parameters', 'pp_plot_b64'])

dict_keys(['distribution', 'parameters', 'pp_plot_b64'])

In [9]:
for result in data['step3']['los_fit_results']['gim'].values():
    del result['los_fit']['pp_plot_b64']
for result in data['step3']['los_fit_results']['icu'].values():
    del result['pre_icu_los_fit']['pp_plot_b64']
    del result['icu_los_fit']['pp_plot_b64']
    del result['post_icu_los_fit']['pp_plot_b64']

print(json.dumps(data['step3']['los_fit_results'], indent=2))

{
  "error": null,
  "gim": {
    "1": {
      "n_patients": 151,
      "probability": 0.1411214953271028,
      "label_groups": {
        "('gim', 'survived', '0-15')": {
          "count": 151,
          "p_total": 0.1411214953271028,
          "p_in_label": 1
        }
      },
      "query": "(ICUAdmission.isnull() and Summary.str.lower() not in ['dead', 'deceased'] and Age >= 0 and Age < 16)",
      "los_fit": {
        "distribution": "Lognormal_3P",
        "parameters": [
          -0.9814889252449772,
          1.5449934357618347,
          0.03654637660200506
        ]
      }
    },
    "2": {
      "n_patients": 398,
      "probability": 0.3719626168224299,
      "label_groups": {
        "('gim', 'survived', '16-64')": {
          "count": 398,
          "p_total": 0.3719626168224299,
          "p_in_label": 1
        }
      },
      "query": "(ICUAdmission.isnull() and Summary.str.lower() not in ['dead', 'deceased'] and Age >= 16 and Age < 65)",
      "los_fit": {
      

## Step 4

In [10]:
data['step4']['jitter']

0

In [11]:
pd.read_feather(BytesIO(b64decode(data['step4']['dailies'])))

,date,count
0,2024-09-20,1
1,2024-09-21,0
2,2024-09-22,1
3,2024-09-23,0
4,2024-09-24,1
...,...,...
245,2025-05-23,0
246,2025-05-24,0
247,2025-05-25,0
248,2025-05-26,0


In [12]:
pd.read_feather(BytesIO(b64decode(data['step4']['hourlies'])))

,hour,probability
0,0,0.0206961
1,1,0.0084666
2,2,0.0131703
3,3,0.0112888
4,4,0.0112888
5,5,0.0244591
6,6,0.0216369
7,7,0.0206961
8,8,0.0206961
9,9,0.0329257


In [13]:
import openpyxl as oxl
wb = oxl.Workbook()
ws = wb.active
# Rename the default sheet
ws.title = "GIM_patients"
# Write the column headers
headers = [
    'Pathway',
    'Label',
    '# Patients',
    'Probability',
    'Outcome',
    'Age',
    'Count',
    '% total',
    '% in label',
    'Distribution',
    'Param1',
    'Param2',
    'Param3',
]
headers = {x[1]: x[0] for x in list(enumerate(headers, start=1))}
# Write the headers to the first row
for header, col_num in headers.items():
    ws.cell(row=1, column=col_num, value=header)
    ws.cell(row=1, column=col_num).font = oxl.styles.Font(bold=True)

def merge_cells(ws, headers, col_name, row_start, n_rows):
    col_num = headers[col_name]
    if n_rows > 1:
        ws.merge_cells(
            start_row=row_start,
            start_column=col_num,
            end_row=row_start + n_rows - 1,
            end_column=col_num
        )

row_num = 2
gim_data = data['step3']['los_fit_results']['gim']
for label, label_data in gim_data.items():
    n_groups = len(label_data['label_groups'])
    row_range = f"{row_num}:{row_num + n_groups - 1}"

    if n_groups > 1:
        merge_cells(ws, headers, 'Pathway', row_num, n_groups)
        merge_cells(ws, headers, 'Label', row_num, n_groups)
        merge_cells(ws, headers, '# Patients', row_num, n_groups)
        merge_cells(ws, headers, 'Probability', row_num, n_groups)
        merge_cells(ws, headers, 'Distribution', row_num, n_groups)
        merge_cells(ws, headers, 'Param1', row_num, n_groups)
        merge_cells(ws, headers, 'Param2', row_num, n_groups)
        merge_cells(ws, headers, 'Param3', row_num, n_groups)
    
    # Write the label-level data to the first row for this label
    ws.cell(row=row_num, column=headers['Pathway'], value='GIM')
    ws.cell(row=row_num, column=headers['Label'], value=label)
    ws.cell(row=row_num, column=headers['# Patients'], value=label_data['n_patients'])

    ws.cell(row=row_num, column=headers['Probability'], value=label_data['probability'])
    ws.cell(row=row_num, column=headers['Probability']).number_format = '0.000%'

    ws.cell(row=row_num, column=headers['Distribution'], value=label_data['los_fit']['distribution'])
    ws.cell(row=row_num, column=headers['Param1'], value=label_data['los_fit']['parameters'][0])
    ws.cell(row=row_num, column=headers['Param2'], value=label_data['los_fit']['parameters'][1])
    ws.cell(row=row_num, column=headers['Param3'], value=label_data['los_fit']['parameters'][2])
    ws.cell(row=row_num, column=headers['Param1']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Param2']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Param3']).number_format = '0.0000'

    # Write the group-level data for each group in the label
    for group, group_data in label_data['label_groups'].items():
        # key is in the format "("gim", '{outcome}', '{age_group}')"
        _, outcome, age_group = [s.strip("'") for s in group.strip('()').split(', ')]
        ws.cell(row=row_num, column=headers['Outcome'], value=outcome)
        ws.cell(row=row_num, column=headers['Age'], value=age_group)
        ws.cell(row=row_num, column=headers['Count'], value=group_data['count'])
        ws.cell(row=row_num, column=headers['% total'], value=group_data['p_total'])
        ws.cell(row=row_num, column=headers['% total']).number_format = '0.000%'
        ws.cell(row=row_num, column=headers['% in label'], value=group_data['p_in_label'])
        ws.cell(row=row_num, column=headers['% in label']).number_format = '0.000%'
        row_num += 1

In [14]:
# Create a new worksheet for ICU patients
ws = wb.create_sheet(title="ICU_patients")

# Write the column headers
headers = [
    'Pathway',
    'Label',
    '# Patients',
    'Probability',
    'Outcome',
    'Age',
    'Count',
    '% total',
    '% in label',
    'Pre_Distribution',
    'Pre_Param1',
    'Pre_Param2',
    'Pre_Param3',
    'Distribution',
    'Param1',
    'Param2',
    'Param3',
    'Post_Distribution',
    'Post_Param1',
    'Post_Param2',
    'Post_Param3',
]
headers = {x[1]: x[0] for x in list(enumerate(headers, start=1))}
# Write the headers to the first row
for header, col_num in headers.items():
    ws.cell(row=1, column=col_num, value=header)
    ws.cell(row=1, column=col_num).font = oxl.styles.Font(bold=True)

def merge_cells(ws, headers, col_name, row_start, n_rows):
    col_num = headers[col_name]
    if n_rows > 1:
        ws.merge_cells(
            start_row=row_start,
            start_column=col_num,
            end_row=row_start + n_rows - 1,
            end_column=col_num
        )

row_num = 2
icu_data = data['step3']['los_fit_results']['icu']
for label, label_data in icu_data.items():
    n_groups = len(label_data['label_groups'])
    row_range = f"{row_num}:{row_num + n_groups - 1}"

    if n_groups > 1:
        merge_cells(ws, headers, 'Pathway', row_num, n_groups)
        merge_cells(ws, headers, 'Label', row_num, n_groups)
        merge_cells(ws, headers, '# Patients', row_num, n_groups)
        merge_cells(ws, headers, 'Probability', row_num, n_groups)
        merge_cells(ws, headers, 'Pre_Distribution', row_num, n_groups)
        merge_cells(ws, headers, 'Pre_Param1', row_num, n_groups)
        merge_cells(ws, headers, 'Pre_Param2', row_num, n_groups)
        merge_cells(ws, headers, 'Pre_Param3', row_num, n_groups)
        merge_cells(ws, headers, 'Distribution', row_num, n_groups)
        merge_cells(ws, headers, 'Param1', row_num, n_groups)
        merge_cells(ws, headers, 'Param2', row_num, n_groups)
        merge_cells(ws, headers, 'Param3', row_num, n_groups)
        merge_cells(ws, headers, 'Post_Distribution', row_num, n_groups)
        merge_cells(ws, headers, 'Post_Param1', row_num, n_groups)
        merge_cells(ws, headers, 'Post_Param2', row_num, n_groups)
        merge_cells(ws, headers, 'Post_Param3', row_num, n_groups)
    
    # Write the label-level data to the first row for this label
    ws.cell(row=row_num, column=headers['Pathway'], value='ICU')
    ws.cell(row=row_num, column=headers['Label'], value=label)
    ws.cell(row=row_num, column=headers['# Patients'], value=label_data['n_patients'])

    ws.cell(row=row_num, column=headers['Probability'], value=label_data['probability'])
    ws.cell(row=row_num, column=headers['Probability']).number_format = '0.000%'

    ws.cell(row=row_num, column=headers['Pre_Distribution'], value=label_data['pre_icu_los_fit']['distribution'])
    ws.cell(row=row_num, column=headers['Pre_Param1'], value=label_data['pre_icu_los_fit']['parameters'][0])
    ws.cell(row=row_num, column=headers['Pre_Param2'], value=label_data['pre_icu_los_fit']['parameters'][1])
    ws.cell(row=row_num, column=headers['Pre_Param3'], value=label_data['pre_icu_los_fit']['parameters'][2])
    ws.cell(row=row_num, column=headers['Pre_Param1']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Pre_Param2']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Pre_Param3']).number_format = '0.0000'

    ws.cell(row=row_num, column=headers['Distribution'], value=label_data['icu_los_fit']['distribution'])
    ws.cell(row=row_num, column=headers['Param1'], value=label_data['icu_los_fit']['parameters'][0])
    ws.cell(row=row_num, column=headers['Param2'], value=label_data['icu_los_fit']['parameters'][1])
    ws.cell(row=row_num, column=headers['Param3'], value=label_data['icu_los_fit']['parameters'][2])
    ws.cell(row=row_num, column=headers['Param1']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Param2']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Param3']).number_format = '0.0000'

    ws.cell(row=row_num, column=headers['Post_Distribution'], value=label_data['post_icu_los_fit']['distribution'])
    ws.cell(row=row_num, column=headers['Post_Param1'], value=label_data['post_icu_los_fit']['parameters'][0])
    ws.cell(row=row_num, column=headers['Post_Param2'], value=label_data['post_icu_los_fit']['parameters'][1])
    ws.cell(row=row_num, column=headers['Post_Param3'], value=label_data['post_icu_los_fit']['parameters'][2])
    ws.cell(row=row_num, column=headers['Post_Param1']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Post_Param2']).number_format = '0.0000'
    ws.cell(row=row_num, column=headers['Post_Param3']).number_format = '0.0000'

    # Write the group-level data for each group in the label
    for group, group_data in label_data['label_groups'].items():
        # key is in the format "("icu", '{outcome}', '{age_group}')"
        _, outcome, age_group = [s.strip("'") for s in group.strip('()').split(', ')]
        ws.cell(row=row_num, column=headers['Outcome'], value=outcome)
        ws.cell(row=row_num, column=headers['Age'], value=age_group)
        ws.cell(row=row_num, column=headers['Count'], value=group_data['count'])
        ws.cell(row=row_num, column=headers['% total'], value=group_data['p_total'])
        ws.cell(row=row_num, column=headers['% total']).number_format = '0.000%'
        ws.cell(row=row_num, column=headers['% in label'], value=group_data['p_in_label'])
        ws.cell(row=row_num, column=headers['% in label']).number_format = '0.000%'
        row_num += 1

In [15]:
ws = wb.create_sheet(title="Sim_params")
ws['A1'] = 'Jitter'
ws['B1'] = data['step4']['jitter'] / 100  # Convert to a proportion for percentage formatting
ws['B1'].number_format = '0%'

In [16]:
from openpyxl.utils.dataframe import dataframe_to_rows as df_to_rows

ws = wb.create_sheet(title="Dailies")
dailies_df = pd.read_feather(BytesIO(b64decode(data['step4']['dailies'])))
for r in df_to_rows(dailies_df, index=False, header=True):
    ws.append(r)

# Bold the header row (only the filled part)
n_cols = len(dailies_df.columns)  # No index so this is just the number of columns in the DataFrame
for col_num in range(1, n_cols + 1):
    ws.cell(row=1, column=col_num).font = oxl.styles.Font(bold=True)

ws = wb.create_sheet(title="Hourlies")
hourlies_df = pd.read_feather(BytesIO(b64decode(data['step4']['hourlies'])))
for r in df_to_rows(hourlies_df, index=False, header=True):
    ws.append(r)

# Bold the header row (only the filled part)
n_cols = len(hourlies_df.columns)  # No index so this is just the number of columns in the DataFrame
for col_num in range(1, n_cols + 1):
    ws.cell(row=1, column=col_num).font = oxl.styles.Font(bold=True)

In [17]:
wb.save('model_export.xlsx')